Sampling of audio file

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip uninstall -y numpy

Found existing installation: numpy 1.23.5
Uninstalling numpy-1.23.5:
  Successfully uninstalled numpy-1.23.5


In [ ]:
!pip install numpy==1.23.5


In [ ]:
!pip install --force-reinstall matplotlib PyEMD


  Using cached pyemd-1.0.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 2.0 MB/s eta 0:00:00
  Using cached numpy-2.2.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/8.6 MB ? eta -:--:--
ERROR: Operation cancelled by user


In [ ]:

import librosa
import numpy as np
import matplotlib.pyplot as plt

# Path to the audio file
file_path = '/content/Audio  3-[AudioTrimmer.com].wav'  # Replace with your file path

# Parameters for sampling
TARGET_SAMPLE_RATE = 22050  # Desired sample rate in Hz

# Load the audio file
audio_data, sample_rate = librosa.load(file_path, sr=TARGET_SAMPLE_RATE)

# Display information about the audio
print(f"Original Sample Rate: {sample_rate}")
print(f"Number of Samples: {len(audio_data)}")
print(f"Duration (seconds): {len(audio_data) / sample_rate}")

# Display a few samples
print("Sampled Data (first 10 samples):", audio_data[:10])

# (Optional) Plot the audio waveform
plt.figure(figsize=(10, 4))
plt.plot(np.linspace(0, len(audio_data) / sample_rate, len(audio_data)), audio_data)
plt.title("Audio Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()


Cleaning of samples

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

# Path to the audio file
file_path = '/content/Audio  3-[AudioTrimmer.com].wav'  # Replace with your file path

# Parameters for sampling
TARGET_SAMPLE_RATE = 22050  # Desired sample rate in Hz

# Load the audio file
audio_data, sample_rate = librosa.load(file_path, sr=TARGET_SAMPLE_RATE)

# Display information about the original audio
print(f"Original Sample Rate: {sample_rate}")
print(f"Number of Samples: {len(audio_data)}")
print(f"Duration (seconds): {len(audio_data) / sample_rate}")

# Step 1: Trim silent parts
trimmed_audio, _ = librosa.effects.trim(audio_data, top_db=20)  # Removes silence below 20 dB
print(f"Number of Samples after Trimming: {len(trimmed_audio)}")
print(f"Duration after Trimming (seconds): {len(trimmed_audio) / sample_rate}")

# Step 2: Noise removal using a bandpass filter
def bandpass_filter(signal, sr, lowcut=100, highcut=3000):
    from scipy.signal import butter, lfilter

    # Design the bandpass filter
    nyquist = 0.5 * sr
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(N=4, Wn=[low, high], btype='band')

    # Apply the filter
    filtered_signal = lfilter(b, a, signal)
    return filtered_signal

filtered_audio = bandpass_filter(trimmed_audio, sample_rate)

# Step 3: Plot the cleaned audio waveform
plt.figure(figsize=(10, 4))
plt.plot(np.linspace(0, len(filtered_audio) / sample_rate, len(filtered_audio)), filtered_audio)
plt.title("Filtered and Trimmed Audio Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()

# Display a few samples of the cleaned audio
print("Cleaned Sampled Data (first 10 samples):", filtered_audio[:10])


In [ ]:
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from PyEMD import EMD  # Install using `pip install EMD-signal`

# Path to the audio file
file_path = '/content/Audio  3-[AudioTrimmer.com].wav'  # Replace with your file path

# Parameters
TARGET_SAMPLE_RATE = 22050  # Desired sample rate in Hz

# Load the audio file
audio_data, sample_rate = librosa.load(file_path, sr=TARGET_SAMPLE_RATE)

# Trim silent parts (optional)
audio_data, _ = librosa.effects.trim(audio_data, top_db=20)

# Function to calculate instantaneous frequency using Hilbert Transform
def instantaneous_frequency(signal, sample_rate):
    # Apply Hilbert Transform to compute the analytic signal
    analytic_signal = hilbert(signal)

    # Compute the instantaneous phase
    instantaneous_phase = np.unwrap(np.angle(analytic_signal))

    # Derive the instantaneous frequency (time derivative of the phase)
    instantaneous_freq = np.diff(instantaneous_phase) / (2.0 * np.pi) * sample_rate

    return instantaneous_freq

# Empirical Mode Decomposition (EMD)
emd = EMD()
imfs = emd(audio_data)

# Plot the IMFs
plt.figure(figsize=(12, 8))
for i, imf in enumerate(imfs):
    plt.subplot(len(imfs), 1, i + 1)
    plt.plot(imf, color='b')
    plt.title(f"IMF {i + 1}")
plt.tight_layout()
plt.show()

# Analyze instantaneous frequency for each IMF
instantaneous_frequencies = []
time_stamps = np.arange(len(audio_data)) / sample_rate

for i, imf in enumerate(imfs):
    # Compute the instantaneous frequency for the IMF
    inst_freq = instantaneous_frequency(imf, sample_rate)

    # Avoid first value (diff reduces one sample)
    time_stamps_imf = time_stamps[:len(inst_freq)]
    instantaneous_frequencies.append((time_stamps_imf, inst_freq))

    # Plot instantaneous frequency
    plt.figure(figsize=(12, 6))
    plt.plot(time_stamps_imf, inst_freq, marker='o', linestyle='-', label=f'IMF {i + 1}')
    plt.title(f"Instantaneous Frequency for IMF {i + 1} (Hilbert Transform)")
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.grid(True)
    plt.legend()
    plt.show()

# Combine information from IMFs (optional)
# Example: Use the first few IMFs with dominant frequencies
dominant_frequencies = []
for time_stamps_imf, inst_freq in instantaneous_frequencies:
    mean_freq = np.mean(inst_freq[inst_freq > 0])  # Ignore negative or zero frequencies
    if mean_freq > 0:
        dominant_frequencies.append(mean_freq)

print("Dominant frequencies (Hz):", dominant_frequencies)


ModuleNotFoundError: No module named 'PyEMD'

Methode-2(FFT)

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq


# Path to the audio file
file_path = '/content/Audio 2-[AudioTrimmer.com].wav'  # Replace with your file path


# Parameters
TARGET_SAMPLE_RATE = 22050  # Desired sample rate in Hz
SEGMENT_DURATION = 0.1      # Segment duration in seconds (0.1s or 100ms)


# Load the audio file
audio_data, sample_rate = librosa.load(file_path, sr=TARGET_SAMPLE_RATE)


# Calculate the number of samples per segment
samples_per_segment = int(SEGMENT_DURATION * sample_rate)


# Function to calculate the dominant frequency of a segment
def dominant_frequency(segment, sample_rate):
    # Perform FFT on the segment
    fft_result = fft(segment)


    # Calculate frequencies corresponding to FFT values
    freqs = fftfreq(len(segment), 1 / sample_rate)


    # Calculate magnitude of FFT values
    magnitudes = np.abs(fft_result)


    # Only consider the positive half of the spectrum (real frequencies)
    positive_freqs = freqs[:len(freqs) // 2]
    positive_magnitudes = magnitudes[:len(magnitudes) // 2]


    # Find the frequency with the highest magnitude
    dominant_freq = positive_freqs[np.argmax(positive_magnitudes)]
    return dominant_freq


# Analyze each segment and find dominant frequencies
dominant_frequencies = []
for i in range(0, len(audio_data), samples_per_segment):
    segment = audio_data[i:i + samples_per_segment]


    # Ensure the segment has the right length
    if len(segment) == samples_per_segment:
        freq = dominant_frequency(segment, sample_rate)
        dominant_frequencies.append(freq)


# Plot the dominant frequencies over time
time_stamps = np.arange(0, len(dominant_frequencies)) * SEGMENT_DURATION
plt.figure(figsize=(10, 4))
plt.plot(time_stamps, dominant_frequencies)
plt.title("Dominant Frequency Over Time")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.show()


# Print the first few dominant frequencies
print("Dominant frequencies in each segment (Hz):", dominant_frequencies[:10])


In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

# Path to the audio file
file_path = '/content/Audio 2-[AudioTrimmer.com].wav'  # Replace with your file path

# Parameters
TARGET_SAMPLE_RATE = 22050  # Desired sample rate in Hz
SEGMENT_DURATION = 0.1      # Segment duration in seconds (0.1s or 100ms)

# Define frequencies of the Indian musical notes (Sa Re Ga Ma Pa Dha Ni Sa)
NOTES = [
    ("Sa", 261.63),  # C4
    ("Re", 293.66),  # D4
    ("Ga", 329.63),  # E4
    ("Ma", 349.23),  # F4
    ("Pa", 392.00),  # G4
    ("Dha", 440.00),  # A4
    ("Ni", 493.88),  # B4
    ("Sa", 523.25),  # C5
]

# Load the audio file
audio_data, sample_rate = librosa.load(file_path, sr=TARGET_SAMPLE_RATE)

# Calculate the number of samples per segment
samples_per_segment = int(SEGMENT_DURATION * sample_rate)

# Function to calculate the dominant frequency of a segment
def dominant_frequency(segment, sample_rate):
    # Perform FFT on the segment
    fft_result = fft(segment)

    # Calculate frequencies corresponding to FFT values
    freqs = fftfreq(len(segment), 1 / sample_rate)

    # Calculate magnitude of FFT values
    magnitudes = np.abs(fft_result)

    # Only consider the positive half of the spectrum (real frequencies)
    positive_freqs = freqs[:len(freqs) // 2]
    positive_magnitudes = magnitudes[:len(magnitudes) // 2]

    # Find the frequency with the highest magnitude
    dominant_freq = positive_freqs[np.argmax(positive_magnitudes)]
    return dominant_freq

# Function to find the nearest musical note
def find_nearest_note(frequency):
    if frequency <= 0:
        return "Silence"
    closest_note = min(NOTES, key=lambda note: abs(note[1] - frequency))
    return closest_note[0]

# Analyze each segment and map to musical notes
detected_notes = []
for i in range(0, len(audio_data), samples_per_segment):
    segment = audio_data[i:i + samples_per_segment]

    # Ensure the segment has the right length
    if len(segment) == samples_per_segment:
        freq = dominant_frequency(segment, sample_rate)
        note = find_nearest_note(freq)
        detected_notes.append(note)

# Plot the detected notes over time
time_stamps = np.arange(0, len(detected_notes)) * SEGMENT_DURATION
plt.figure(figsize=(10, 4))
plt.scatter(time_stamps, detected_notes, marker='o', color='b')
plt.title("Detected Notes Over Time")
plt.xlabel("Time (s)")
plt.ylabel("Note")
plt.grid()
plt.show()

# Print the detected notes
print("Detected notes:", detected_notes)


Methode-3(Peak Detection)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.io.wavfile import read

def detect_frequencies(audio_file, threshold=0.1):
    # Read the audio file
    sample_rate, audio_data = read(audio_file)

    # Ensure audio is mono by selecting only one channel if it's stereo
    if len(audio_data.shape) > 1:
        audio_data = audio_data[:, 0]

    # Perform FFT
    n = len(audio_data)
    fft_result = np.fft.fft(audio_data)
    fft_magnitude = np.abs(fft_result[:n // 2])  # Take the positive half of the spectrum
    frequencies = np.fft.fftfreq(n, d=1/sample_rate)[:n // 2]

    # Detect peaks in the FFT magnitude
    peaks, _ = find_peaks(fft_magnitude, height=threshold * max(fft_magnitude))

    # Extract peak frequencies
    peak_frequencies = frequencies[peaks]
    peak_magnitudes = fft_magnitude[peaks]

    # Print detected frequencies and their magnitudes
    print("Detected Frequencies:")
    for freq, mag in zip(peak_frequencies, peak_magnitudes):
        print(f"{freq:.2f} Hz - Magnitude: {mag:.2f}")

    # Plot the FFT with detected peaks
    plt.figure(figsize=(10, 6))
    plt.plot(frequencies, fft_magnitude, label="FFT Magnitude")
    plt.plot(peak_frequencies, peak_magnitudes, "r*", label="Detected Peaks")
    plt.title("Frequency Spectrum with Peak Detection")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")
    plt.legend()
    plt.grid()
    plt.show()

# Example usage
audio_file_path = "/content/Audio 2-[AudioTrimmer.com].wav"  # Replace with your audio file path
detect_frequencies(audio_file_path, threshold=0.1)


Methode-4(Cepstrum Analysis)

In [ ]:
import numpy as np
from scipy.io import wavfile
from scipy.signal import get_window
import matplotlib.pyplot as plt

def compute_cepstrum(signal, sample_rate, window_type='hamming'):
    """
    Computes the cepstrum of a signal.

    Parameters:
    - signal: Audio signal array.
    - sample_rate: Sample rate of the signal.
    - window_type: Type of window to apply to the signal.

    Returns:
    - real_cepstrum: The real cepstrum of the signal.
    - quefrency: The quefrency array corresponding to the cepstrum.
    """
    window = get_window(window_type, len(signal))
    signal_windowed = signal * window
    spectrum = np.fft.fft(signal_windowed)
    log_spectrum = np.log(np.abs(spectrum) + 1e-10)  # Avoid log(0)
    real_cepstrum = np.fft.ifft(log_spectrum).real
    quefrency = np.arange(len(real_cepstrum)) / sample_rate
    return real_cepstrum, quefrency

def find_fundamental_frequency(cepstrum, quefrency, min_f0=50, max_f0=500):
    """
    Finds the fundamental frequency of a signal using its cepstrum.

    Parameters:
    - cepstrum: Real cepstrum of the signal.
    - quefrency: Quefrency array corresponding to the cepstrum.
    - min_f0: Minimum possible fundamental frequency (Hz).
    - max_f0: Maximum possible fundamental frequency (Hz).

    Returns:
    - fundamental_frequency: Detected fundamental frequency (Hz).
    """
    min_quefrency = 1 / max_f0
    max_quefrency = 1 / min_f0
    quefrency_range = (quefrency >= min_quefrency) & (quefrency <= max_quefrency)

    peak_index = np.argmax(cepstrum[quefrency_range])
    fundamental_quefrency = quefrency[quefrency_range][peak_index]
    fundamental_frequency = 1 / fundamental_quefrency
    return fundamental_frequency

def process_audio(file_path):
    # Read audio file
    sample_rate, signal = wavfile.read(file_path)
    if signal.ndim > 1:  # If stereo, take only one channel
        signal = signal[:, 0]

    # Normalize signal
    signal = signal / np.max(np.abs(signal))

    # Compute cepstrum
    cepstrum, quefrency = compute_cepstrum(signal, sample_rate)

    # Find fundamental frequency
    fundamental_frequency = find_fundamental_frequency(cepstrum, quefrency)

    print(f"Fundamental Frequency: {fundamental_frequency:.2f} Hz")

    # Plotting (Optional)
    plt.figure(figsize=(10, 6))
    plt.plot(quefrency, cepstrum)
    plt.title("Cepstrum")
    plt.xlabel("Quefrency (s)")
    plt.ylabel("Amplitude")
    plt.grid()
    plt.show()

# Example usage
audio_file_path = "/content/Audio 2-[AudioTrimmer.com].wav"  # Replace with your audio file path
process_audio(audio_file_path)


Notation Display by Peak Detaction methode

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.io.wavfile import read

# Define a function to map frequency to the closest musical note
def frequency_to_note_name(freq):
    if freq == 0:
        return "Silence"

    # Define A4 reference frequency and note names
    A4 = 440.0
    note_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

    # Calculate the number of semitones from A4
    semitones_from_A4 = 12 * np.log2(freq / A4)
    rounded_semitones = int(round(semitones_from_A4))

    # Find the note index and octave
    note_index = (rounded_semitones + 9) % 12  # A is index 9 in note_names
    octave = (rounded_semitones + 9) // 12 + 4  # A4 is in octave 4

    return f"{note_names[note_index]}{octave}"

# Detect frequencies and map to notes
def detect_frequencies_and_notes(audio_file, threshold=0.1):
    # Read the audio file
    sample_rate, audio_data = read(audio_file)

    # Ensure audio is mono by selecting only one channel if it's stereo
    if len(audio_data.shape) > 1:
        audio_data = audio_data[:, 0]

    # Perform FFT
    n = len(audio_data)
    fft_result = np.fft.fft(audio_data)
    fft_magnitude = np.abs(fft_result[:n // 2])  # Take the positive half of the spectrum
    frequencies = np.fft.fftfreq(n, d=1/sample_rate)[:n // 2]

    # Detect peaks in the FFT magnitude
    peaks, _ = find_peaks(fft_magnitude, height=threshold * max(fft_magnitude))

    # Extract peak frequencies
    peak_frequencies = frequencies[peaks]
    peak_magnitudes = fft_magnitude[peaks]

    # Map frequencies to notes
    notes = [frequency_to_note_name(freq) for freq in peak_frequencies]

    # Print detected notes and their corresponding frequencies
    print("Detected Notes and Frequencies:")
    for freq, note in zip(peak_frequencies, notes):
        print(f"{freq:.2f} Hz - {note}")

    # Plot the FFT with detected peaks
    plt.figure(figsize=(10, 6))
    plt.plot(frequencies, fft_magnitude, label="FFT Magnitude")
    plt.plot(peak_frequencies, peak_magnitudes, "r*", label="Detected Peaks")
    for freq, note in zip(peak_frequencies, notes):
        plt.text(freq, fft_magnitude[frequencies == freq][0], note, color='red', fontsize=9, ha='center')
    plt.title("Frequency Spectrum with Detected Notes")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")
    plt.legend()
    plt.grid()
    plt.show()

# Example usage
audio_file_path = "/content/Audio 2-[AudioTrimmer.com].wav"  # Replace with your audio file path
detect_frequencies_and_notes(audio_file_path, threshold=0.1)


Notation Display by FFT & Hilbert

In [ ]:
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Path to the audio file
file_path = '/content/Audio 2-[AudioTrimmer.com].wav'  # Replace with your file path

# Parameters
TARGET_SAMPLE_RATE = 22050  # Desired sample rate in Hz

# Load the audio file
audio_data, sample_rate = librosa.load(file_path, sr=TARGET_SAMPLE_RATE)

# Trim silent parts (optional)
audio_data, _ = librosa.effects.trim(audio_data, top_db=20)

# Define swara frequencies (in Hz)
swara_frequencies = {
    "Sa": 261.63,   # C4
    "Re": 293.66,   # D4
    "Ga": 329.63,   # E4
    "Ma": 349.23,   # F4
    "Pa": 392.00,   # G4
    "Dha": 440.00,  # A4
    "Ni": 493.88,   # B4
    "Sa'": 523.25   # C5 (higher octave)
}

# Function to find the closest swara for a given frequency
def get_closest_swara(frequency, swara_frequencies):
    closest_swara = None
    min_diff = float('inf')

    for swara, swara_freq in swara_frequencies.items():
        diff = abs(frequency - swara_freq)
        if diff < min_diff:
            min_diff = diff
            closest_swara = swara

    return closest_swara

# Function to calculate instantaneous frequency using Hilbert Transform
def instantaneous_frequency(signal, sample_rate):
    # Apply Hilbert Transform to compute the analytic signal
    analytic_signal = hilbert(signal)

    # Compute the instantaneous phase
    instantaneous_phase = np.unwrap(np.angle(analytic_signal))

    # Derive the instantaneous frequency (time derivative of the phase)
    instantaneous_freq = np.diff(instantaneous_phase) / (2.0 * np.pi) * sample_rate

    return instantaneous_freq

# Parameters for frame-based analysis
SEGMENT_DURATION = 0.1  # Segment duration in seconds (100ms)
samples_per_segment = int(SEGMENT_DURATION * sample_rate)

# Analyze instantaneous frequency for each segment
instantaneous_frequencies = []
swaras = []
time_stamps = []

for i in range(0, len(audio_data) - samples_per_segment, samples_per_segment):
    segment = audio_data[i:i + samples_per_segment]
    inst_freq = instantaneous_frequency(segment, sample_rate)
    time_stamps.append(i / sample_rate)

    if len(inst_freq) > 0:  # Ensure the segment has a valid frequency
        mean_inst_freq = np.mean(inst_freq)  # Use the mean frequency for the segment
        instantaneous_frequencies.append(mean_inst_freq)

        # Map the instantaneous frequency to the closest swara
        swara = get_closest_swara(mean_inst_freq, swara_frequencies)
        swaras.append(swara)
    else:
        instantaneous_frequencies.append(0)
        swaras.append("Rest")  # Use "Rest" for silent or invalid segments

# Plot the instantaneous frequencies and swaras over time
plt.figure(figsize=(12, 6))
plt.plot(time_stamps, instantaneous_frequencies, marker='o', linestyle='-', color='b', label="Instantaneous Frequency (Hz)")

# Annotate swaras on the graph
for i, (time, swara) in enumerate(zip(time_stamps, swaras)):
    plt.text(time, instantaneous_frequencies[i], swara, fontsize=9, rotation=45, ha="right", va="bottom")

plt.title("Swaras Over Time (Hilbert Transform)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.grid(True)
plt.legend()
plt.show()

# Print the detected swaras over time
print("Swaras detected for each segment:")
for time, swara in zip(time_stamps, swaras):
    print(f"Time: {time:.2f}s - Swara: {swara}")
